In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
# Add the main directory to sys.path
sys.path.append(main_dir)
# importing
from OptFunctions.TwoDBox import *
from Box_functions import *
from TwoD_EGPA import *
import warnings
warnings.filterwarnings('ignore')
import time
import pickle

In [2]:
x = np.linspace(0, 1, 20)
y = np.linspace(0, 1, 20)
dx = x[-1]-x[-2]
dy = y[-1]-y[-2]

Cartesianstack = None
for i in range(len(x)):
    for j in range(len(y)):
        cartesian = np.array((x[i], y[j]))
        cartesian = np.reshape(cartesian, (1, 2)) 
        if Cartesianstack is None:
            Cartesianstack=cartesian
        else:
            Cartesianstack = np.concatenate([Cartesianstack, cartesian]) 

# Extract frontier points
frontier_mask = (Cartesianstack[:, 0] == 0) | (Cartesianstack[:, 0] == 1) | (Cartesianstack[:, 1] == 0) | (Cartesianstack[:, 1] == 1)
### Crear datos para fijar evidencias en proceso gaussiano
X_init = Cartesianstack[frontier_mask]
Y_init = np.zeros((len(X_init), 1))
init_data = (X_init, Y_init)

THPsi0flat, THPsi0, THEintegrando = Psinxny(1,1, x=x, y=y, dx=dx, dy=dy)
THPsi1flat, THPsi1, THE1integrando = Psinxny(1,2,x=x, y=y, dx=dx, dy=dy)
THPsi12flat, THPsi12, THE12integrando = Psinxny(2,1,x=x, y=y, dx=dx, dy=dy)
THPsi2flat, THPsi2, THE2integrando = Psinxny(2, 2, x=x, y=y, dx=dx, dy=dy)
THPsi3flat, THPsi3, THE3integrando = Psinxny(3, 1, x=x, y=y, dx=dx, dy=dy)
THPsi31flat, THPsi31, THE31integrando = Psinxny(1, 3, x=x, y=y, dx=dx, dy=dy)

######## Esto hará falta cuando se hagan los experimentos y se calcule el error cometido en la predicción del GP a puntos fuera del grid de cálculo #################
GPx = np.linspace(0, 1, 32)
GPy = np.linspace(0, 1, 32)
dGPx = GPx[-1] - GPx[-2]
dGPy = GPy[-1] - GPy[-2]
GPstack = None
for i in range(len(GPx)):
    for j in range(len(GPy)):
        cartesian = np.array((GPx[i], GPy[j]))
        cartesian = np.reshape(cartesian, (1, 2)) 
        if GPstack is None:
            GPstack=cartesian
        else:
            GPstack = np.concatenate([GPstack, cartesian]) 

GPTHPsi0flat, GPTHPsi0, GPTHEintegrando = Psinxny(1,1, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
GPTHPsi1flat, GPTHPsi1, GPTHE1integrando = Psinxny(1,2,x=GPx, y=GPy, dx=dGPx, dy=dGPy)
GPTHPsi12flat, GPTHPsi12, GPTHE12integrando = Psinxny(2,1,x=GPx, y=GPy, dx=dGPx, dy=dGPy)
GPTHPsi2flat, GPTHPsi2, GPTHE2integrando = Psinxny(2, 2, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
GPTHPsi3flat, GPTHPsi3, GPTHE3integrando = Psinxny(3, 1, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
GPTHPsi31flat, GPTHPsi31, GPTHE31integrando = Psinxny(1, 3, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
###############################################################################################################################


In [3]:
#First we import the optimized algorithm hyperparameters

ruta = f'BO_params/BOparameters'
with open(f'{ruta}.pkl', "rb") as file:
    BO_params = pickle.load(file)


size_gen = BO_params['params']['size_gen']
alpha = BO_params['params']['alpha']
elite_factor = BO_params['params']['elite_factor']
init_kernels = BO_params['params']['init_kernels']
std_lower_bound = BO_params['params']['std_lower_bound']


#### STOP CRITERIA ##### based on max number of function evaluations fixed at 1 x 10^6
max_f_evals = 1*10**6 # max_f_evals = max_iter * size_gen.  
max_iter = int(max_f_evals/size_gen)
dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter

Nexp = 10
Resultados = np.empty((Nexp, 4, 5)) # 10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
print(f'\n\n########################### Experimentos con 400 variables ###########################\n\n')
for experimento in range(Nexp):
    while True:
            try:
                print(f'\n\n########################### Ejecución de Experimentos número: {experimento+1} de {Nexp} ###########################\n\n')
                print(f'Estado 11')
                ########################## ESTADO 11 ##########################
                params = {}
                params['dx'] = dx
                params['dy'] = dy
                Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
                start = time.time()
                egparesultground = Gaussian.minimize(TwoDBox, params, exp_from = 300, output_runtime=True)
                end = time.time()
                Resultados[experimento, 0, 4] = end - start #10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]

                Estado = np.reshape(egparesultground.data[-1, :-1, 0], (len(egparesultground.data[-1, :-1, 0]), 1))
                Psi0= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
                E0 = EnergyBox(Psi0, dx=dx, dy=dy)
                Resultados[experimento, 0, 1] = E0
                Resultados[experimento, 0, 2] = abs(E0-Enxny(1,1))/Enxny(1,1)*100
                MSE0 = min((np.mean((egparesultground.data[-1, :-1, 0] - THPsi0flat)**2),
                            np.mean((-1*egparesultground.data[-1, :-1, 0] - THPsi0flat)**2)))
                Resultados[experimento, 0, 0] = MSE0

                #Last fitted GP prediction for outside points is:
                GPprediction = egparesultground.GP_hist.predict(GPstack)
                Estado = np.reshape(GPprediction, (len(GPprediction), 1))
                GPPsi0= reconvert_points_from_cartesian_stack_to_2D(Estado, x=GPx, y=GPy)
                NGPPsi0 = Normalize(GPPsi0, dx=dGPx, dy=dGPy)
                #NGPE0 = EnergyBox(NGPPsi0, dx=dGPx, dy=dGPy)
                NGPMSE0 = min((np.mean((NGPPsi0 - GPTHPsi0)**2),
                        np.mean((-1*NGPPsi0 - GPTHPsi0)**2)))
                Resultados[experimento, 0, 3] = NGPMSE0 # 10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]

                print(f'La energía del estado 1,1 teórica es: {Enxny(1,1)},\nEGPA encuentra energía: {E0},\ncon un error respecto de la solución teórica discretizada MSE= {MSE0}')
                print(f'\n######## Estado 11 DONE ########\n')

                print(f'Estado 12')
                ###################### ESTADO 12 o 21 ##############################
                params = {}
                params['dx'] = dx
                params['dy'] = dy
                params['Psi0'] = Psi0
                params['E0'] = E0
                Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
                start = time.time()
                egparesultexc1 = Gaussian.minimize(TwoDBoxExc1, params, exp_from = 420, output_runtime=True)
                end = time.time()
                Resultados[experimento, 1, 4] = end - start #10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                Estado = np.reshape(egparesultexc1.data[-1, :-1, 0], (len(egparesultexc1.data[-1, :-1, 0]), 1))
                Psi1= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
                E1 = EnergyBox(Psi1, dx=dx, dy=dy)
                Resultados[experimento, 1, 1] = E1
                Resultados[experimento, 1, 2] = abs(E1-Enxny(1,2))/Enxny(1,2)*100
                MSE1 = min(np.mean((egparesultexc1.data[-1, :-1, 0] - THPsi1flat)**2),
                    np.mean((-1*egparesultexc1.data[-1, :-1, 0] - THPsi1flat)**2),
                    np.mean((egparesultexc1.data[-1, :-1, 0] - THPsi12flat)**2),
                    np.mean((-1*egparesultexc1.data[-1, :-1, 0] - THPsi12flat)**2))
                Resultados[experimento, 1, 0] = MSE1
                #Last fitted GP prediction for outside points is:
                GPprediction = egparesultexc1.GP_hist.predict(GPstack)
                Estado = np.reshape(GPprediction, (len(GPprediction), 1))
                GPPsi1= reconvert_points_from_cartesian_stack_to_2D(Estado, x=GPx, y=GPy)
                NGPPsi1 = Normalize(GPPsi1, dx=dGPx, dy=dGPy)
                #GPE1 = EnergyBox(GPPsi1, dx=dGPx, dy=dGPy)
                #NGPE1 = EnergyBox(NGPPsi1, dx=dGPx, dy=dGPy)
                #GPMSE1 = min((np.mean((GPprediction - GPTHPsi1flat)**2),np.mean((-1*GPprediction - GPTHPsi1flat)**2)))
                NGPMSE1 = min(np.mean((NGPPsi1 - GPTHPsi1)**2),
                        np.mean((-1*NGPPsi1 - GPTHPsi1)**2),
                        np.mean((NGPPsi1 - GPTHPsi12)**2),
                        np.mean((-1*NGPPsi1 - GPTHPsi12)**2))
                Resultados[experimento, 1, 3] = NGPMSE1 # 10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                print(f'\n######## Estado 12 DONE ########\n')

                print(f'Estado 21')
                ###################### ESTADO ORTOGONAL a 12 o 21 ##############################
                params = {}
                params['dx'] = dx
                params['dy'] = dy
                params['Psi0'] = Psi0
                params['E0'] = E0
                params['Psi1'] = Psi1
                params['E1'] = E1

                Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True,init_data=init_data, normalize_solutions=True)
                start = time.time()
                egparesultexc12 = Gaussian.minimize(TwoDBoxExc12, params, exp_from = 420, output_runtime=True)
                end = time.time()
                Resultados[experimento, 2, 4] = end - start #10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                Estado = np.reshape(egparesultexc12.data[-1, :-1, 0], (len(egparesultexc12.data[-1, :-1, 0]), 1))
                Psi12= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
                E12 = EnergyBox(Psi12, dx=dx, dy=dy)
                Resultados[experimento, 2, 1] = E12
                Resultados[experimento, 2, 2] = abs(E12-Enxny(1,2))/Enxny(1,2)*100
                MSE12 = min(np.mean((egparesultexc12.data[-1, :-1, 0] - THPsi1flat)**2),
                            np.mean((-1*egparesultexc12.data[-1, :-1, 0] - THPsi1flat)**2),
                            np.mean((egparesultexc12.data[-1, :-1, 0] - THPsi12flat)**2),
                            np.mean((-1*egparesultexc12.data[-1, :-1, 0] - THPsi12flat)**2))
                Resultados[experimento, 2, 0] = MSE12

                #El proceso gaussiano último es:
                GPprediction = egparesultexc12.GP_hist.predict(GPstack)
                Estado = np.reshape(GPprediction, (len(GPprediction), 1))
                GPPsi12= reconvert_points_from_cartesian_stack_to_2D(Estado, x=GPx, y=GPy)
                NGPPsi12 = Normalize(GPPsi12, dx=dGPx, dy=dGPy)
                #GPE12 = EnergyBox(GPPsi12, dx=dGPx, dy=dGPy)
                #NGPE12 = EnergyBox(NGPPsi12, dx=dGPx, dy=dGPy)
                #GPMSE12 = min((np.mean((GPprediction - GPTHPsi12flat)**2), np.mean((-1*GPprediction - GPTHPsi12flat)**2)))
                NGPMSE12 = min(np.mean((NGPPsi12 - GPTHPsi12)**2),
                                 np.mean((-1*NGPPsi12 - GPTHPsi12)**2),
                                 np.mean((NGPPsi12 - GPTHPsi1)**2),
                                 np.mean((-1*NGPPsi12 - GPTHPsi1)**2))
                Resultados[experimento, 2, 3] = NGPMSE12 # 10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                print(f'\n######## Estado 21 DONE ########\n')

                
                print(f'Estado 22')
                ###################### ESTADO 22 ##############################
                params = {}
                params['dx'] = dx
                params['dy'] = dy
                params['Psi0'] = Psi0
                params['E0'] = E0
                params['Psi1'] = Psi1
                params['E1'] = E1
                params['Psi12'] = Psi12
                params['E12'] = E12

                Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
                start = time.time()
                egparesultexc2 = Gaussian.minimize(TwoDBoxExc2, params, exp_from = 403, output_runtime=True)
                end = time.time()
                Resultados[experimento, 3, 4] = end - start #10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                Estado = np.reshape(egparesultexc2.data[-1, :-1, 0], (len(egparesultexc2.data[-1, :-1, 0]), 1))
                Psi2= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
                E2 = EnergyBox(Psi2, dx=dx, dy=dy)
                Resultados[experimento, 3, 1] = E2
                Resultados[experimento, 3, 2] = abs(E2-Enxny(2,2))/Enxny(2,2)*100
                MSE2 = min(np.mean((egparesultexc2.data[-1, :-1, 0] - THPsi2flat)**2),
                            np.mean((-1*egparesultexc2.data[-1, :-1, 0] - THPsi2flat)**2))
                Resultados[experimento, 3, 0] = MSE2

                #El proceso gaussiano último es:
                GPprediction = egparesultexc2.GP_hist.predict(GPstack)
                Estado = np.reshape(GPprediction, (len(GPprediction), 1))
                GPPsi2= reconvert_points_from_cartesian_stack_to_2D(Estado, x=GPx, y=GPy)
                NGPPsi2 = Normalize(GPPsi2, dx=dGPx, dy=dGPy)
                #GPE2 = EnergyBox(GPPsi2, dx=dGPx, dy=dGPy)
                #NGPE2 = EnergyBox(NGPPsi2, dx=dGPx, dy=dGPy)
                # GPMSE2 = min((np.mean((GPprediction - GPTHPsi2flat)**2),
                #             np.mean((-1*GPprediction - GPTHPsi2flat)**2)))
                NGPMSE2 = min((np.mean((NGPPsi2 - GPTHPsi2)**2),
                            np.mean((-1*NGPPsi2 - GPTHPsi2)**2)))
                Resultados[experimento, 3, 3] = NGPMSE12 # 10 exps, 4estados, 5 métricas [MSE, E, E%, GP_1000, t_exc]
                print(f'\n######## Estado 2 DONE ########\n')
                print(f'\n######## Experimento {experimento+1} de 10 completado')
                break
            except:
                 print('Ha fallado el ajuste del kernel en algún momento. Repitiendo experimento.')
                 continue
##################################################################################################

rutafinal = f'EGPA2D_results/results'    
with open(f'{rutafinal}.pkl', "wb") as file:
    pickle.dump(Resultados, file)



########################### Experimentos con 400 variables ###########################




########################### Ejecución de Experimentos número: 1 de 10 ###########################


Estado 11
IT:  0 	Best cost:  97.79232554322203
IT:  1 	Best cost:  96.60885674803632
IT:  2 	Best cost:  96.60885674803632
IT:  3 	Best cost:  96.60885674803632
IT:  4 	Best cost:  96.60885674803632
IT:  5 	Best cost:  96.60885674803632
IT:  6 	Best cost:  96.60885674803632
IT:  7 	Best cost:  96.60885674803632
IT:  8 	Best cost:  96.60885674803632
IT:  9 	Best cost:  96.60885674803632
IT:  10 	Best cost:  96.60885674803632
IT:  11 	Best cost:  96.60885674803632
IT:  12 	Best cost:  96.60885674803632
IT:  13 	Best cost:  96.60885674803632
IT:  14 	Best cost:  96.60885674803632
IT:  15 	Best cost:  96.60885674803632
IT:  16 	Best cost:  96.60885674803632
IT:  17 	Best cost:  96.60885674803632
IT:  18 	Best cost:  96.60885674803632
IT:  19 	Best cost:  96.60885674803632
IT:  20 	Best cost:  96.608